# Stroke Prediction Model

This notebook has been cleaned up for modern Python and pandas/scikit-learn compatibility. It keeps the original analysis workflow while replacing outdated patterns and deprecated APIs.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## Load dataset

The dataset is stored under the `data/` folder in the repository root.

In [ ]:
data_path = Path.cwd() / 'data' / 'healthcare-dataset-stroke-data.csv'
df = pd.read_csv(data_path)
df.head()


## Data cleaning and feature preparation

- Remove the single `Other` gender row because it is not representative for modeling.
- Fill missing BMI values with the median.
- Fill missing smoking status with `Unknown`.
- Keep the target column as `stroke`.

In [ ]:
df = df[df['gender'] != 'Other'].copy()
df['bmi'] = df['bmi'].fillna(df['bmi'].median())
df['smoking_status'] = df['smoking_status'].fillna('Unknown')

target = 'stroke'
numeric_features = [
    'age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi'
]
categorical_features = [
    'gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status'
]

X = df[numeric_features + categorical_features]
y = df[target]

X.head()


## Train/test split

Use stratification to preserve the class balance in the target variable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape


## Build a modern scikit-learn pipeline

This replaces old ad-hoc preprocessing with a cleaner `Pipeline` + `ColumnTransformer` setup.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
)

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model),
    ]
)

pipeline.fit(X_train, y_train)


## Evaluate the model

Classification performance is assessed with precision, recall, F1-score, and ROC-AUC.

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f'ROC AUC: {roc_auc_score(y_test, y_proba):.4f}')
print('Confusion Matrix:
', confusion_matrix(y_test, y_pred))


## Optional: feature importance view

A quick way to inspect the most influential features when using a logistic model.

In [ ]:
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
coefficients = pipeline.named_steps['model'].coef_[0]
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
}).sort_values('coefficient', key=lambda s: s.abs(), ascending=False)

feature_importance.head(10)


### Notes

- This notebook avoids deprecated pandas and scikit-learn patterns such as `as_matrix`, `iteritems`, older encoder APIs, and direct `plot_confusion_matrix` use.
- The code is written to work with modern pandas 2.x and scikit-learn 1.4+ releases.